In [35]:
%pip install -U google-generativeai
%pip install google-ai-generativelanguage==0.6.15
%pip install -U langchain-google-genai
%pip install -U langchain-community
%pip install -U langgraph
%pip install google-genai


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Thiago\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Thiago\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Thiago\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Thiago\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Thiago\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [36]:
import os 
import re 
from google import genai
from langgraph.graph import StateGraph, END 
from typing import TypedDict

In [37]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI_API_KEY') 
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

TypeError: str expected, not NoneType

In [44]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",  # mais seguro
    contents="Hello world"
)

print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Hello to you too! How can I help you today?


In [50]:
class Agent:
    def __init__(self, system=""):
        self.history = []
        if system:
            self.history.append(f"Sistema: {system}")

    def __call__(self, message):
        self.history.append(f"Usuário: {message}")

        result = self.execute()

        self.history.append(f"Assistente: {result}")

        return result

    def execute(self):
        response = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents="\n".join(self.history)  # 🔥 aqui está a correção
        )

        return response.text


if __name__ == "__main__":
    agent = Agent("Você é um assistente útil.")
    print(agent("Qual é a capital da França?"))

A capital da França é **Paris**.


In [17]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.
""".strip()


In [51]:
class AgentState(TypedDict):
    pergunta: str
    historico: list[str]
    acao_pendente: str
    resposta_final: str

In [19]:
def consultar_estoque(item: str) -> str:
    """
    Simula a consulta de estoque de um item no inventário.
    """
    item = item.lower()
    estoque = {
        "monitor": 75,
        "teclado": 120,
        "mouse gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impressora": 15
    }

    if item in estoque:
        return f"Temos {estoque[item]} {item}s em estoque."
    else:
        return f"Item '{item}' não encontrado no inventário."

def consultar_preco_produto(produto: str) -> str:
    """
    Simula a consulta do preço unitário de um produto.
    """
    produto = produto.lower()
    precos = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if produto in precos:
        return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
    else:
        return f"Produto '{produto}' não encontrado na lista de preços."



In [20]:
print(consultar_estoque("teclado"))
print(consultar_preco_produto("impressora"))
print(consultar_estoque("monitor"))

Temos 120 teclados em estoque.
O preço de um(a) impressora é R$ 750.00.
Temos 75 monitors em estoque.


In [21]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    model = genai.GenerativeModel('gemini-2.5-flash')
    chat = model.start_chat(history=[])

    chat.send_message(PROMPT_REACT)

    current_prompt = pergunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        if response_text.startswith("Resposta:"):
            return response_text.replace("Resposta:", "").strip()

        match = re.search(r"Ação:\s*(\w+):\s*(.*)", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip()

            observacao = ""
            if action_name == "consultar_estoque":
                observacao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao = consultar_preco_produto(action_arg)
            else:
                observacao = f"Erro: Ação '{action_name}' desconhecida."

            current_prompt = f"Observação: {observacao}\nResposta:"

            print(f"Executou ação: {action_name}({action_arg})")
            print(f"Observação: {observacao}\n")

        else:
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta: {response_text}"

    return "Erro: Limite máximo de iterações atingido sem uma resposta final."

In [22]:
pergunta_1 = "Quantos mouses gamers estão no inventário?"
print(f"**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")

print("\n" + "="*50 + "\n") 


**Interação 1: Quantos mouses gamers estão no inventário?**
--- Iteração 1 ---
Modelo pensou/respondeu:
Pensamento: O usuário está perguntando sobre a quantidade de "mouses gamers" no inventário. Devo usar a ferramenta `consultar_estoque` para obter essa informação.
Ação: consultar_estoque: mouse gamer
PAUSA

Executou ação: consultar_estoque(mouse gamer)
Observação: Temos 80 mouse gamers em estoque.

--- Iteração 2 ---
Modelo pensou/respondeu:
Observação: Temos 80 mouse gamers em estoque.
Resposta: Há 80 mouses gamers em estoque.


**RESPOSTA FINAL DO AGENTE 1:** Erro: O agente não conseguiu extrair uma Ação ou Resposta final após 2 iterações. Última resposta: Observação: Temos 80 mouse gamers em estoque.
Resposta: Há 80 mouses gamers em estoque.





In [23]:

pergunta_2 = "Qual o valor de uma impressora?"
print(f"**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")

print("\n" + "="*50 + "\n") 



**Interação 2: Qual o valor de uma impressora?**
--- Iteração 1 ---
Modelo pensou/respondeu:
Pensamento: Devo consultar a ação consultar_preco_produto para saber o valor de uma impressora.
Ação: consultar_preco_produto: impressora
PAUSA

Executou ação: consultar_preco_produto(impressora)
Observação: O preço de um(a) impressora é R$ 750.00.

--- Iteração 2 ---
Modelo pensou/respondeu:
Resposta: O preço de uma impressora é R$ 750,00.


**RESPOSTA FINAL DO AGENTE 2:** O preço de uma impressora é R$ 750,00.





In [24]:
pergunta_3 = "Temos cadeiras em estoque?"
print(f"**Interação 3: {pergunta_3}**")
resposta_3 = run_react_agent(pergunta_3)
print(f"\n**RESPOSTA FINAL DO AGENTE 3:** {resposta_3}\n")

**Interação 3: Temos cadeiras em estoque?**
--- Iteração 1 ---
Modelo pensou/respondeu:
Pensamento: Para saber se há cadeiras em estoque, preciso consultar o estoque de cadeiras.
Ação: consultar_estoque: cadeira
PAUSA

Executou ação: consultar_estoque(cadeira)
Observação: Item 'cadeira' não encontrado no inventário.

--- Iteração 2 ---
Modelo pensou/respondeu:
Resposta: Não foi possível verificar o estoque de cadeiras, pois o item 'cadeira' não foi encontrado no inventário.


**RESPOSTA FINAL DO AGENTE 3:** Não foi possível verificar o estoque de cadeiras, pois o item 'cadeira' não foi encontrado no inventário.



In [25]:
pergunta_4 = "Qual é o produto mais caro?"
print(f"**Interação 4: {pergunta_4}**")
resposta_4 = run_react_agent(pergunta_4)
print(f"\n**RESPOSTA FINAL DO AGENTE 4:** {resposta_4}\n")

**Interação 4: Qual é o produto mais caro?**
--- Iteração 1 ---
Modelo pensou/respondeu:
Pensamento: Para determinar o produto mais caro, eu precisaria ter uma lista de todos os produtos disponíveis e seus respectivos preços, ou a capacidade de consultar o preço de vários produtos e depois compará-los. As ações disponíveis (consultar_estoque e consultar_preco_produto) funcionam para produtos específicos. Não possuo uma ferramenta para listar todos os produtos ou para comparar automaticamente os preços de produtos múltiplos e desconhecidos. Portanto, não consigo responder a esta pergunta diretamente com as ferramentas que me foram fornecidas.
Resposta: Não consigo determinar qual é o produto mais caro, pois não tenho uma ferramenta para listar todos os produtos ou para comparar seus preços automaticamente sem saber quais produtos verificar.


**RESPOSTA FINAL DO AGENTE 4:** Erro: O agente não conseguiu extrair uma Ação ou Resposta final após 1 iterações. Última resposta: Pensamento: Par

In [26]:
def consultar_estoque(item: str) -> str: 
    """
    Simula a consulta de estoque de um item no inventário.
    """
    item = item.lower().strip()
    estoque = {
        "monitor": 75,
        "teclado": 120,
        "mouse gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impressora": 15
    }

    if item in estoque:
        return f"Temos {estoque[item]} {item}s em estoque."
    else:
        return f"Item '{item}' não encontrado no inventário."

def consultar_preco_produto(produto: str) -> str: 
    """
    Simula a consulta do preço unitário de um produto.
    """
    produto = produto.lower().strip()
    precos = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if produto in precos:
        return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
    else:
        return f"Produto '{produto}' não encontrado na lista de preços."

def ferramenta_encontrar_produto_mais_caro() -> str: 
    """
    Retorna o nome e o preço do produto mais caro no inventário.
    Esta função não precisa de argumentos.
    """
    
    precos_do_inventario = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if not precos_do_inventario: 
        return "Nenhum produto encontrado na lista de preços para comparação."

    
    nome_produto_mais_caro = max(precos_do_inventario, key=precos_do_inventario.get)
    valor_produto_mais_caro = precos_do_inventario[nome_produto_mais_caro]

    return f"O produto mais caro é o(a) {nome_produto_mais_caro} com preço de R$ {valor_produto_mais_caro:.2f}." 




In [27]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")
  - encontrar_produto_mais_caro: retorna o nome e o preço do produto mais caro no inventário (não requer argumentos)

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.

Exemplo:
Pergunta: Qual é o produto mais caro?
Pensamento: Preciso usar a ação encontrar_produto_mais_caro para descobrir qual produto tem o maior preço.
Ação: encontrar_produto_mais_caro
PAUSA

Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.
Resposta: O produto mais caro é o(a) monitor com preço de R$ 999.90.
""".strip()

In [60]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:

    history = []
    history.append(f"Sistema: {PROMPT_REACT}")

    current_prompt = pergunta

    for i in range(max_iterations):

        history.append(f"Usuário: {current_prompt}")

        response = client.models.generate_content(
            model="gemini-3.1-flash-lite-preview",
            contents="\n".join(history)
        )

        response_text = response.text.strip()

        history.append(f"Assistente: {response_text}")

        print(f"\n--- Iteração {i+1} ---")
        print(f"Modelo:\n{response_text}\n")

        # ✅ resposta final
        final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if final:
            return final.group(1).strip()

        # ✅ ação
        match = re.search(r"Ação:\s*([^\n:]+)(?::\s*(.*))?", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) else ""

            if action_name == "consultar_estoque":
                obs = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                obs = consultar_preco_produto(action_arg)
            elif action_name == "encontrar_produto_mais_caro":
                obs = ferramenta_encontrar_produto_mais_caro()
            else:
                obs = f"Ação desconhecida: {action_name}"

            print(f"Executou: {action_name}({action_arg})")
            print(f"Observação: {obs}\n")

            current_prompt = f"Observação: {obs}\nContinue."

        else:
            return f"Erro de parsing:\n{response_text}"

    return "Erro: limite de iterações atingido."

In [61]:
print("--- Começando as Interações com o Agente ReAct ---")

# Interação 1: Consultar Estoque
pergunta_1 = "Quantos teclados temos em estoque?"
print(f"\n**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")

print("\n" + "="*80 + "\n")

# Interação 2: Consultar Preço
pergunta_2 = "Qual o preço de um headset?"
print(f"\n**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")

print("\n" + "="*80 + "\n")

# Interação 3: Item não encontrado no estoque
pergunta_3 = "Temos cadeiras em estoque?"
print(f"\n**Interação 3: {pergunta_3}**")
resposta_3 = run_react_agent(pergunta_3)
print(f"\n**RESPOSTA FINAL DO AGENTE 3:** {resposta_3}\n")

print("\n" + "="*80 + "\n")

# Interação 4: Encontrar o Produto Mais Caro (A nova funcionalidade!)
pergunta_4 = "Qual é o produto mais caro?"
print(f"\n**Interação 4: {pergunta_4}**")
resposta_4 = run_react_agent(pergunta_4)
print(f"\n**RESPOSTA FINAL DO AGENTE 4:** {resposta_4}\n")

print("\n--- Fim das Interações ---")

--- Começando as Interações com o Agente ReAct ---

**Interação 1: Quantos teclados temos em estoque?**

--- Iteração 1 ---
Modelo:
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de teclados disponíveis no inventário.
Ação: consultar_estoque: teclado
PAUSA

Executou: consultar_estoque(teclado)
Observação: Temos 120 teclados em estoque.


--- Iteração 2 ---
Modelo:
Resposta: Temos 120 teclados em estoque.


**RESPOSTA FINAL DO AGENTE 1:** Temos 120 teclados em estoque.




**Interação 2: Qual o preço de um headset?**

--- Iteração 1 ---
Modelo:
Pensamento: Preciso consultar o preço de um "headset" utilizando a ferramenta apropriada.
Ação: consultar_preco_produto: headset
PAUSA

Executou: consultar_preco_produto(headset)
Observação: O preço de um(a) headset é R$ 180.00.


--- Iteração 2 ---
Modelo:
Resposta: O preço de um headset é R$ 180,00.


**RESPOSTA FINAL DO AGENTE 2:** O preço de um headset é R$ 180,00.




**Interação 3: Temos cadeiras em estoque?**



In [59]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025

In [ ]:
def run_react_agent_with_history(pergunta: str, max_iterations: int = 5) -> tuple[str, list]: 
    model = genai.GenerativeModel('gemini-2.5-flash') 

    chat = model.start_chat(history=[])
    chat.send_message(PROMPT_REACT) 

    current_prompt = pergunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"\n--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        response_match_final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            return response_match_final.group(1).strip(), chat.history

        match = re.search(r"Ação:\s*(\w+)(?::\s*([^\n]*))?", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            observacao_da_acao = ""

            if action_name == "consultar_estoque":
                observacao_da_acao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao_da_acao = consultar_preco_produto(action_arg)
            elif action_name == "encontrar_produto_mais_caro":
                observacao_da_acao = ferramenta_encontrar_produto_mais_caro()
            else:
                observacao_da_acao = f"Erro: Ação '{action_name}' desconhecida. Verifique o prompt ou a implementação da ferramenta."

            current_prompt = f"Observação: {observacao_da_acao}"

            print(f"Executou ação: {action_name} com argumento '{action_arg}'")
            print(f"Observação: {observacao_da_acao}\n")

        else:
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta do modelo: {response_text}", chat.history

    return "Erro: Limite máximo de iterações atingido sem uma resposta final do agente.", chat.history

In [ ]:
pergunta_exemplo = "Quantos teclados temos em estoque?"
resposta_exemplo, historico_completo = run_react_agent_with_history(pergunta_exemplo)

print(f"**RESPOSTA FINAL DO AGENTE:** {resposta_exemplo}\n")

print("\n--- Histórico Completo da Interação ---")
for i, message in enumerate(historico_completo):
    print(f"--- Mensagem {i+1} (role: {message.role}) ---")

    for part in message.parts:
        if hasattr(part, 'text'):
            print(part.text)
        else:
            print(part)
    print("-" * 20)
print("\n--- Fim do Histórico ---\n")

In [62]:
def ferramenta_calcular_valor_total_lista(lista_itens: str) -> str:
    """
    Calcula o valor total de uma lista de itens de compra.
    Recebe uma string com itens separados por vírgula (ex: "teclado, mouse gamer, monitor").
    """
    precos_do_inventario = { 
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    itens_processados = [item.strip().lower() for item in lista_itens.split(',')]

    valor_total = 0.0
    itens_nao_encontrados = []


    for item in itens_processados: 
        if item in precos_do_inventario:
            valor_total += precos_do_inventario[item]
        else:
            itens_nao_encontrados.append(item)

    resposta = f"O valor total dos itens encontrados é R$ {valor_total:.2f}."
    if itens_nao_encontrados:
        resposta += f" Os seguintes itens não foram encontrados e não foram incluídos no cálculo: {', '.join(itens_nao_encontrados)}."

    return resposta


In [63]:
print("Testando ferramenta_calcular_valor_total_lista:")

lista_1 = "teclado, mouse gamer, monitor"
resultado_1 = ferramenta_calcular_valor_total_lista(lista_1)
print(f"Lista: '{lista_1}'\nResultado: {resultado_1}\n")


Testando ferramenta_calcular_valor_total_lista:
Lista: 'teclado, mouse gamer, monitor'
Resultado: O valor total dos itens encontrados é R$ 1249.40.



In [64]:
lista_2 = "headset, cadeira"
resultado_2 = ferramenta_calcular_valor_total_lista(lista_2)
print(f"Lista: '{lista_2}'\nResultado: {resultado_2}\n")


Lista: 'headset, cadeira'
Resultado: O valor total dos itens encontrados é R$ 180.00. Os seguintes itens não foram encontrados e não foram incluídos no cálculo: cadeira.



In [65]:
lista_3 = "mesa, copo"
resultado_3 = ferramenta_calcular_valor_total_lista(lista_3)
print(f"Lista: '{lista_3}'\nResultado: {resultado_3}\n")


Lista: 'mesa, copo'
Resultado: O valor total dos itens encontrados é R$ 0.00. Os seguintes itens não foram encontrados e não foram incluídos no cálculo: mesa, copo.



In [66]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    
    history = []
    history.append(f"Sistema: {PROMPT_REACT}")

    current_prompt = pergunta

    for i in range(max_iterations):

        history.append(f"Usuário: {current_prompt}")

        response = client.models.generate_content(
            model="gemini-3.1-flash-lite-preview",
            contents="\n".join(history)
        )

        response_text = response.text.strip()

        history.append(f"Assistente: {response_text}")

        print(f"\n--- Iteração {i+1} ---")
        print(f"Modelo:\n{response_text}\n")

        # ✅ resposta final
        final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if final:
            return final.group(1).strip()

        # ✅ ação
        match = re.search(r"Ação:\s*([^\n:]+)(?::\s*(.*))?", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            observacao_da_acao = ""

            if action_name == "consultar_estoque":
                observacao_da_acao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao_da_acao = consultar_preco_produto(action_arg)
            elif action_name == "encontrar_produto_mais_caro":
                observacao_da_acao = ferramenta_encontrar_produto_mais_caro()
            elif action_name == "calcular_valor_total_lista":
                observacao_da_acao = ferramenta_calcular_valor_total_lista(action_arg)
            else:
                observacao_da_acao = f"Erro: Ação '{action_name}' desconhecida. Verifique o prompt ou a implementação da ferramenta."

            current_prompt = f"Observação: {observacao_da_acao}"

            print(f"Executou ação: {action_name} com argumento '{action_arg}'")
            print(f"Observação: {observacao_da_acao}\n")

        else:
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta do modelo: {response_text}"

    return "Erro: Limite máximo de iterações atingido sem uma resposta final do agente."

In [67]:
print("--- Começando as Interações com o Agente ReAct ---")

# Interação 1: Consultar Estoque
pergunta_1 = "Quantos teclados temos em estoque?"
print(f"\n**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")

print("\n" + "="*80 + "\n")

# Interação 2: Consultar Preço
pergunta_2 = "Qual o preço de um headset?"
print(f"\n**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")

print("\n" + "="*80 + "\n")

# Interação 3: Item não encontrado no estoque
pergunta_3 = "Temos cadeiras em estoque?"
print(f"\n**Interação 3: {pergunta_3}**")
resposta_3 = run_react_agent(pergunta_3)
print(f"\n**RESPOSTA FINAL DO AGENTE 3:** {resposta_3}\n")

print("\n" + "="*80 + "\n")

# Interação 4: Encontrar o Produto Mais Caro
pergunta_4 = "Qual é o produto mais caro?"
print(f"\n**Interação 4: {pergunta_4}**")
resposta_4 = run_react_agent(pergunta_4)
print(f"\n**RESPOSTA FINAL DO AGENTE 4:** {resposta_4}\n")

print("\n" + "="*80 + "\n")

# Interação 5: Calcular Valor Total da Lista (NOVA FUNCIONALIDADE)
pergunta_5 = "Qual o valor de um teclado, uma impressora e uma webcam?"
print(f"\n**Interação 5: {pergunta_5}**")
resposta_5 = run_react_agent(pergunta_5)
print(f"\n**RESPOSTA FINAL DO AGENTE 5:** {resposta_5}\n")


print("\n--- Fim das Interações ---")

--- Começando as Interações com o Agente ReAct ---

**Interação 1: Quantos teclados temos em estoque?**

--- Iteração 1 ---
Modelo:
Pensamento: Preciso consultar a quantidade de teclados disponíveis no estoque. Para isso, utilizarei a ferramenta consultar_estoque.
Ação: consultar_estoque: teclado
PAUSA

Executou ação: consultar_estoque com argumento 'teclado'
Observação: Temos 120 teclados em estoque.


--- Iteração 2 ---
Modelo:
Resposta: Temos 120 teclados em estoque.


**RESPOSTA FINAL DO AGENTE 1:** Temos 120 teclados em estoque.




**Interação 2: Qual o preço de um headset?**

--- Iteração 1 ---
Modelo:
Pensamento: Devo consultar o preço do headset utilizando a ação consultar_preco_produto.

Ação: consultar_preco_produto: headset
PAUSA

Executou ação: consultar_preco_produto com argumento 'headset'
Observação: O preço de um(a) headset é R$ 180.00.


--- Iteração 2 ---
Modelo:
Resposta: O preço de um headset é R$ 180,00.


**RESPOSTA FINAL DO AGENTE 2:** O preço de um headset é R$

In [68]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")
  - encontrar_produto_mais_caro: retorna o nome e o preço do produto mais caro no inventário (não requer argumentos)
  - calcular_valor_total_lista: calcula o valor total de uma lista de itens de compra. Recebe uma string com itens separados por vírgula (ex: "teclado, mouse gamer, monitor")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.

Exemplo:
Pergunta: Qual é o produto mais caro?
Pensamento: Preciso usar a ação encontrar_produto_mais_caro para descobrir qual produto tem o maior preço.
Ação: encontrar_produto_mais_caro
PAUSA

Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.
Resposta: O produto mais caro é o(a) monitor com preço de R$ 999.90.

Exemplo:
Pergunta: Quanto custa um teclado e um mouse gamer?
Pensamento: O usuário quer saber o valor total de vários itens. Devo usar a ação calcular_valor_total_lista com os itens "teclado, mouse gamer".
Ação: calcular_valor_total_lista: teclado, mouse gamer
PAUSA

Observação: O valor total dos itens encontrados é R$ 249.50.
Resposta: O valor total do teclado e do mouse gamer é R$ 249.50.
""".strip()


In [69]:
print("--- Começando as Interações com o Agente ReAct ---")

# Interação 1: Consultar Estoque
pergunta_1 = "Quantos teclados temos em estoque?"
print(f"\n**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")

print("\n" + "="*80 + "\n")

# Interação 2: Consultar Preço
pergunta_2 = "Qual o preço de um headset?"
print(f"\n**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")

print("\n" + "="*80 + "\n")

# Interação 3: Item não encontrado no estoque
pergunta_3 = "Temos cadeiras em estoque?"
print(f"\n**Interação 3: {pergunta_3}**")
resposta_3 = run_react_agent(pergunta_3)
print(f"\n**RESPOSTA FINAL DO AGENTE 3:** {resposta_3}\n")

print("\n" + "="*80 + "\n")

# Interação 4: Encontrar o Produto Mais Caro
pergunta_4 = "Qual é o produto mais caro?"
print(f"\n**Interação 4: {pergunta_4}**")
resposta_4 = run_react_agent(pergunta_4)
print(f"\n**RESPOSTA FINAL DO AGENTE 4:** {resposta_4}\n")

print("\n" + "="*80 + "\n")

# Interação 5: Calcular Valor Total da Lista (NOVA FUNCIONALIDADE)
pergunta_5 = "Qual o valor de um teclado, uma impressora e uma webcam?"
print(f"\n**Interação 5: {pergunta_5}**")
resposta_5 = run_react_agent(pergunta_5)
print(f"\n**RESPOSTA FINAL DO AGENTE 5:** {resposta_5}\n")


print("\n--- Fim das Interações ---")

--- Começando as Interações com o Agente ReAct ---

**Interação 1: Quantos teclados temos em estoque?**

--- Iteração 1 ---
Modelo:
Pensamento: Preciso verificar a quantidade disponível de teclados no estoque utilizando a ferramenta apropriada.
Ação: consultar_estoque: teclado
PAUSA

Executou ação: consultar_estoque com argumento 'teclado'
Observação: Temos 120 teclados em estoque.


--- Iteração 2 ---
Modelo:
Resposta: Temos 120 teclados em estoque.


**RESPOSTA FINAL DO AGENTE 1:** Temos 120 teclados em estoque.




**Interação 2: Qual o preço de um headset?**

--- Iteração 1 ---
Modelo:
Pensamento: Preciso saber o preço unitário de um headset. Para isso, devo utilizar a ferramenta `consultar_preco_produto`.

Ação: consultar_preco_produto: headset
PAUSA

Executou ação: consultar_preco_produto com argumento 'headset'
Observação: O preço de um(a) headset é R$ 180.00.


--- Iteração 2 ---
Modelo:
Resposta: O preço de um headset é R$ 180.00.


**RESPOSTA FINAL DO AGENTE 2:** O preço de u

In [70]:
def iniciar_conversacao_com_agente():
    print("--- Agente de Inventário Interativo ---")
    print("Digite sua pergunta sobre o inventário, ou digite 'sair' para encerrar.")
    print("-" * 50)

    while True:
        pergunta_usuario = input("\nVocê: ")

        if pergunta_usuario.lower().strip() == 'sair':
            print("Encerrando a conversa. Até logo!")
            break

        print("\nAgente: Processando...")
        try:

            resposta_agente = run_react_agent(pergunta_usuario)
            print(f"\nAgente: {resposta_agente}")
        except Exception as e:

            print(f"\nAgente: Ocorreu um erro ao processar sua pergunta: {e}")
            print("Por favor, tente novamente ou digite 'sair'.")

if __name__ == "__main__":
    iniciar_conversacao_com_agente()

--- Agente de Inventário Interativo ---
Digite sua pergunta sobre o inventário, ou digite 'sair' para encerrar.
--------------------------------------------------

Agente: Processando...

--- Iteração 1 ---
Modelo:
Pensamento: O usuário deseja saber a quantidade de teclados disponíveis no inventário. Devo utilizar a ação consultar_estoque.
Ação: consultar_estoque: teclado
PAUSA

Executou ação: consultar_estoque com argumento 'teclado'
Observação: Temos 120 teclados em estoque.


--- Iteração 2 ---
Modelo:
Resposta: Temos 120 teclados em estoque.


Agente: Temos 120 teclados em estoque.

Agente: Processando...

--- Iteração 1 ---
Modelo:
Pensamento: O usuário deseja saber o valor total de três itens específicos: monitor, impressora e webcam. Devo utilizar a ferramenta `calcular_valor_total_lista` com esses itens.

Ação: calcular_valor_total_lista: monitor, impressora, webcam
PAUSA

Executou ação: calcular_valor_total_lista com argumento 'monitor, impressora, webcam'
Observação: O valor 

KeyboardInterrupt: Interrupted by user